# OddBall NetFlow analysis

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'oddball' / 'data.py').exists())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from oddball import analysis, data, figures, scoring
from oddball.analysis import COMPOSITE_LABELS, COMPOSITE_WEIGHTS
from oddball.data import WEIGHT_COLS, WEIGHT_LABELS

pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 60)

In [ ]:
FILES = [PROJECT_ROOT / f'netflow_day-{day:02d}.bz2' for day in range(3, 33)]
WINDOW_HOUR = 0
TOP_K = 20
LOF_K_RANGE = (10, 50)
RESULTS_DIR = PROJECT_ROOT / 'results'
OUTDIR = RESULTS_DIR / 'descriptive_hour00'
FIGDIR = OUTDIR / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

def show(name):
    display(Image(filename=str(FIGDIR / f'{name}.png')))

## Relative-hour coverage

In [ ]:
coverage_path = RESULTS_DIR / 'activity_by_relative_hour' / 'daily_24_hour_coverage.csv'
subprocess.run([sys.executable, '-m', 'oddball.coverage', '--data-dir', str(PROJECT_ROOT), '--start-day', '2', '--end-day', '33', '--output', str(coverage_path)], check=True)
display(pd.read_csv(coverage_path))

## Day 3 relative-hour-0 graph and edge weights

In [ ]:
flows = data.load_window(FILES[0], 0, 3600, relative=True)
edges = data.build_edge_table(flows)
nodes = pd.unique(pd.concat([edges['src'], edges['dst']], ignore_index=True))
edge_summary = edges[WEIGHT_COLS].describe(percentiles=[0.25, 0.5, 0.75]).T[['min', '25%', '50%', '75%', 'mean', 'max']]
edge_summary.columns = ['minimum', 'q1', 'median', 'q3', 'mean', 'maximum']
edge_summary.index = ['Flow count', 'Packet count', 'Byte count']
edge_summary.index.name = 'weight'
edge_summary.to_csv(OUTDIR / 'primary_edge_weight_summary.csv')
display(pd.DataFrame({'biflows': [len(flows)], 'hosts': [len(nodes)], 'edges': [len(edges)]}))
display(edge_summary)

In [ ]:
correlation = figures.fig_weight_correlation(edges, FIGDIR)
correlation.to_csv(OUTDIR / 'edge_weight_spearman_correlation.csv')
display(correlation.round(3))
show('fig_weight_correlation')

## Single-weight OddBall results

In [ ]:
scored = analysis.score_all_weightings(edges, WEIGHT_COLS, k=LOF_K_RANGE, include_eigenvalue=True)
fit_rows = []
composition_rows = []
top20_rows = []
for law, xcol, ycol in [('EDPL', 'N', 'E'), ('EWPL', 'E', 'W'), ('ELWPL', 'W', 'lambda_w')]:
    prefix = law.lower()
    for weight, frame in scored.items():
        if law == 'EDPL' and weight != 'n_flows':
            continue
        r2, _, bins = scoring.fit_r2(frame[xcol].to_numpy(float), frame[ycol].to_numpy(float))
        fit_rows.append({'law': law, 'weight': '-' if law == 'EDPL' else WEIGHT_LABELS[weight], 'C': frame.attrs[f'{prefix}_C'], 'theta': frame.attrs[f'{prefix}_theta'], 'binned_median_r2': r2, 'bins': bins})
        top = frame.nlargest(TOP_K, f'{prefix}_score')
        composition_rows.append({'law': law, 'weight': '-' if law == 'EDPL' else WEIGHT_LABELS[weight], 'heavy': int((top[f'{prefix}_side'] == 'heavy').sum()), 'light': int((top[f'{prefix}_side'] == 'light').sum()), 'line_driven': int((top[f'{prefix}_score_driver'] == 'line-dominant').sum()), 'lof_driven': int(top[f'{prefix}_score_driver'].str.startswith('LOF').sum())})
        for rank, (_, row) in enumerate(top.iterrows(), 1):
            top20_rows.append({'law': law, 'weight': '-' if law == 'EDPL' else WEIGHT_LABELS[weight], 'rank': rank, 'node': row['node'], 'score': row[f'{prefix}_score'], 'line_component': row[f'{prefix}_line_component'], 'lof_component': row[f'{prefix}_lof_component'], 'side': row[f'{prefix}_side'], 'driver': row[f'{prefix}_score_driver']})

single_fits = pd.DataFrame(fit_rows)
score_composition = pd.DataFrame(composition_rows)
single_top20 = pd.DataFrame(top20_rows)
single_fits.to_csv(OUTDIR / 'primary_single_weight_fits.csv', index=False)
score_composition.to_csv(OUTDIR / 'primary_top20_score_composition.csv', index=False)
single_top20.to_csv(OUTDIR / 'appendix_single_top20.csv', index=False)
display(single_fits.round(3))
display(score_composition)
for (law, weight), table in single_top20.groupby(['law', 'weight'], sort=False):
    title = law if law == 'EDPL' else f'{weight.title()}-weighted {law}'
    view = table[['rank', 'node', 'score', 'line_component', 'lof_component', 'side', 'driver']].copy()
    view['driver'] = view['driver'].map({'line-dominant': 'Line', 'LOF-dominant': 'LOF'})
    view.columns = ['Rank', 'Host', 'Total score', 'Line component', 'LOF component', 'Side', 'Dominant component']
    display(Markdown(f'### {title} Top-20'))
    display(view.round(3))
figures.fig_all_power_laws(scored, FIGDIR, topk=TOP_K)
show('fig_all_power_laws')

In [ ]:
agreement_rows = []
for law in ('EWPL', 'ELWPL'):
    overlap = analysis.cross_weight_overlap(scored, law=law, K=TOP_K).set_index('pair')
    rank_corr = analysis.rank_correlation(scored, law=law)
    for pair in overlap.index:
        first, second = pair.split('-')
        agreement_rows.append({'law': law, 'weight_pair': pair, 'spearman': rank_corr.loc[first, second], 'top20_jaccard': overlap.loc[pair, 'jaccard']})
agreement = pd.DataFrame(agreement_rows)
agreement.to_csv(OUTDIR / 'primary_weighted_law_comparisons.csv', index=False)
display(agreement.round(3))

k_results = analysis.k_curve(scored, law='EWPL')
k_results.to_csv(OUTDIR / 'k_curve.csv', index=False)
figures.fig_k_curve(k_results, FIGDIR)
show('fig_k_curve')

## Stability of the single-weight fits

In [ ]:
windows = [{'name': f'Day {day}', 'file': path, 'start': 0, 'end': 3600, 'relative': True} for day, path in zip(range(3, 33), FILES)]
temporal_fits = analysis.temporal_replication(windows, k_lof=LOF_K_RANGE, laws=('EDPL', 'EWPL', 'ELWPL'))
temporal_summary = analysis.summarise_temporal(temporal_fits)
temporal_fits.to_csv(OUTDIR / 'temporal_fits.csv', index=False)
temporal_summary.to_csv(OUTDIR / 'temporal_fit_summary.csv', index=False)
display(temporal_summary.round(3))

## Composite-weight EWPL results

In [ ]:
display(pd.DataFrame.from_dict(COMPOSITE_WEIGHTS, orient='index', columns=['flow', 'packets', 'bytes']))
composite_edges = analysis.geometric_composite_edges(edges, COMPOSITE_WEIGHTS)
composite_scored = analysis.score_ewpl_weightings_parallel(composite_edges, list(COMPOSITE_WEIGHTS), k=LOF_K_RANGE, processes=10)
composite_summary = analysis.composite_ewpl_summary(composite_scored, COMPOSITE_LABELS, top_k=TOP_K)
composite_top20_rows = []
for method, frame in composite_scored.items():
    for rank, (_, row) in enumerate(frame.nlargest(TOP_K, 'ewpl_score').iterrows(), 1):
        composite_top20_rows.append({'method': method, 'composite': COMPOSITE_LABELS[method].removeprefix('Geometric '), 'rank': rank, 'node': row['node'], 'score': row['ewpl_score'], 'line_component': row['ewpl_line_component'], 'lof_component': row['ewpl_lof_component'], 'side': row['ewpl_side'], 'driver': row['ewpl_score_driver']})
composite_top20 = pd.DataFrame(composite_top20_rows)
composite_summary.to_csv(OUTDIR / 'primary_composite_ewpl_summary.csv', index=False)
composite_top20.to_csv(OUTDIR / 'appendix_composite_top20.csv', index=False)
display(composite_summary.round(3))
for method in COMPOSITE_WEIGHTS:
    table = composite_top20[composite_top20['method'] == method]
    view = table[['rank', 'node', 'score', 'line_component', 'lof_component', 'side', 'driver']].copy()
    view['driver'] = view['driver'].map({'line-dominant': 'Line', 'LOF-dominant': 'LOF'})
    view.columns = ['Rank', 'Host', 'Total score', 'Line component', 'LOF component', 'Side', 'Dominant component']
    display(Markdown(f'### {COMPOSITE_LABELS[method]} EWPL Top-20'))
    display(view.round(3))
figures.fig_composite_ewpl(composite_scored, COMPOSITE_LABELS, FIGDIR, topk=TOP_K)
show('fig_composite_ewpl')

## Controlled replacement experiment

In [ ]:
HOURS = [0, 8, 16]
for hour in HOURS:
    result_dir = RESULTS_DIR / 'replacement' / f'hour{hour:02d}'
    subprocess.run([sys.executable, '-m', 'oddball.replacement', '--data-dir', str(PROJECT_ROOT), '--history-start-day', '3', '--history-end-day', '32', '--test-day', '33', '--start', str(hour * 3600), '--end', str((hour + 1) * 3600), '--history-top-k', '100', '--top-k', '50', '--trials', '100', '--seed', '2026', '--lof-k-min', '10', '--lof-k-max', '50', '--history-processes', '4', '--processes', '14', '--outdir', str(result_dir)], check=True)

In [ ]:
thresholds, pools, cutoffs, paired, clean = [], [], [], [], []
for hour in HOURS:
    result_dir = RESULTS_DIR / 'replacement' / f'hour{hour:02d}'
    label = f'{hour:02d}:00-{hour + 1:02d}:00'
    threshold = pd.read_csv(result_dir / 'historical_activity_thresholds.csv')
    threshold.insert(0, 'window', label)
    thresholds.append(threshold)
    host_pool = pd.read_csv(result_dir / 'historical_host_pool.csv')
    metadata = json.loads((result_dir / 'settings.json').read_text())
    pools.append({'window': label, 'historical_pool': len(host_pool), 'day33_present': metadata['test_day_historical_pool_present'], 'donors': metadata['test_day_donor_candidates'], 'recipients': metadata['test_day_recipient_candidates'], 'trials': metadata['trials']})
    cutoff = pd.read_csv(result_dir / 'cutoff_sensitivity.csv')
    cutoff.insert(0, 'window', label)
    cutoffs.append(cutoff)
    for k in (10, 20, 50):
        frame = pd.read_csv(result_dir / f'paired_top{k}_vs_flow.csv')
        frame.insert(0, 'window', label)
        paired.append(frame)
    status = pd.read_csv(result_dir / 'selected_host_clean_rank_status.csv')
    status.insert(0, 'window', label)
    clean.append(status)

threshold_table = pd.concat(thresholds, ignore_index=True)
pool_table = pd.DataFrame(pools)
cutoff_data = pd.concat(cutoffs, ignore_index=True)
paired_data = pd.concat(paired, ignore_index=True)
clean_status = pd.concat(clean, ignore_index=True)
display(threshold_table)
display(pool_table)
display(clean_status.groupby('window')[['clean_top10', 'clean_top20', 'clean_top50']].sum())

In [ ]:
top20 = cutoff_data[cutoff_data['cutoff'] == 20].pivot(index=['method', 'label'], columns='window', values='post_rate').reset_index()
hour_columns = [f'{hour:02d}:00-{hour + 1:02d}:00' for hour in HOURS]
top20['mean'] = top20[hour_columns].mean(axis=1)
top20['sd'] = top20[hour_columns].std(axis=1)
budget = cutoff_data.groupby(['method', 'label', 'cutoff'])['post_rate'].agg(['mean', 'std']).reset_index()
flow_mean = float(top20.loc[top20['method'] == 'flow', 'mean'].iloc[0])
selected = set(top20.loc[(top20['method'].str.startswith('geo_')) & (top20['mean'] > flow_mean), 'method'])
paired_table = paired_data[paired_data['method'].isin(selected)][['window', 'cutoff', 'method', 'label', 'composite_only', 'baseline_only']]

threshold_table.to_csv(OUTDIR / 'report_activity_thresholds.csv', index=False)
pool_table.to_csv(OUTDIR / 'report_replacement_pools.csv', index=False)
top20.to_csv(OUTDIR / 'report_top20_detection_rates.csv', index=False)
budget.to_csv(OUTDIR / 'report_alert_budget_sensitivity.csv', index=False)
paired_table.to_csv(OUTDIR / 'report_paired_comparison_with_flow.csv', index=False)
display(top20)
display(budget)
display(paired_table)